# 原 Stage1 双概率交互：风险上限释放0bp局部价值信息门

固定 alpha×(1−tail) 单一交互，对照两概率加自身状态及训练均值；使用同起点 Tiered 物理局部标签。

In [1]:
import hashlib,json,os,pathlib,shlex
import numpy as np,pandas as pd,qlib
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from src.analysis.seed_disagreement_information import full_calendar_bootstrap
from src.data.nas_market_data import archive_nas_recorder,_settings,_ssh
root=pathlib.Path.cwd();card=root/'research/stage1_alpha_tail_cap_release_information_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];sha=lambda p:hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and plan['execution_authorized_by_active_Goal']
label_card=root/src['label_card'];frame_card=root/src['frame_card'];assert sha(label_card/'summary.json')==src['label_summary_sha256'] and sha(frame_card/'summary.json')==src['frame_summary_sha256'] and sha(root/'src/analysis/seed_disagreement_information.py')==src['bootstrap_module_sha256']
for card_root,eid,rid,name,key in [(label_card,src['label_experiment_id'],src['label_recorder_id'],'analysis/labels','label_artifact_sha256'),(frame_card,src['frame_experiment_id'],src['frame_recorder_id'],'inputs/source_frame','frame_artifact_sha256'),(frame_card,src['frame_experiment_id'],src['frame_recorder_id'],'paths/reference_0','path_artifact_sha256')]:assert sha(card_root/'qlib_recorder'/eid/rid/'artifacts'/name)==src[key]
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/'data/qlib/full_path_tiered_allocation_20261013'),region='us',kernels=1)
with R.uri_context(str(frame_card/'qlib_recorder')):
 rec=R.get_recorder(experiment_id=src['frame_experiment_id'],recorder_id=src['frame_recorder_id']);assert rec.status=='FINISHED';frame=rec.load_object('inputs/source_frame').reset_index(drop=True);path=rec.load_object('paths/reference_0')['replay'].reset_index(drop=True)
with R.uri_context(str(label_card/'qlib_recorder')):
 rec=R.get_recorder(experiment_id=src['label_experiment_id'],recorder_id=src['label_recorder_id']);assert rec.status=='FINISHED';labels=rec.load_object('analysis/labels').reset_index(drop=True)
assert len(frame)==len(path)==len(labels)==1061 and frame.trade_date.equals(path.trade_date) and frame.trade_date.equals(labels.trade_date)
assert (pd.to_datetime(frame.feature_available_at,utc=True)<=pd.to_datetime(labels.decision_at,utc=True)).all()
for name in ('alpha_probability','tail_probability'):
 assert np.isfinite(frame[name].to_numpy(float)).all() and frame[name].between(0,1).all()
flat=pd.DataFrame({'signal_date':pd.to_datetime(frame.signal_date),'trade_date':pd.to_datetime(frame.trade_date),'alpha_probability':frame.alpha_probability.astype(float),'tail_probability':frame.tail_probability.astype(float),'raw_beta_minus_cap':labels.raw_beta_minus_cap.astype(float),'risk_state':path.risk_state.astype(float),'interaction':frame.alpha_probability.astype(float)*(1-frame.tail_probability.astype(float)),'local_fee_net_advantage':labels.local_fee_net_advantage.astype(float),'label_available_at':pd.to_datetime(labels.label_available_at,utc=True),'decision_at':pd.to_datetime(labels.decision_at,utc=True),'eligible':labels.eligible.astype(bool),'cap_binding':labels.raw_beta_minus_cap.gt(1e-8).astype(bool)})
assert not flat.signal_date.duplicated().any() and flat.signal_date.is_monotonic_increasing
assert (flat.loc[flat.eligible,'cap_binding']).all() and int(flat.eligible.sum())==541
features=plan['fixed_features']['baseline']+['interaction'];static=pd.concat({'feature':flat[features],'label':flat[['local_fee_net_advantage']],'meta':flat[['label_available_at','decision_at','eligible','cap_binding']]},axis=1);static.index=pd.MultiIndex.from_arrays([flat.trade_date,['SOXL_SOXX_CASH']*len(flat)],names=['datetime','instrument'])
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'train':('2018-10-15','2020-12-31'),'evaluation':('2021-01-01','2022-12-30')})
tr=dataset.prepare('train',col_set=['feature','label','meta']);ev=dataset.prepare('evaluation',col_set=['feature','label','meta']);tr_dates=pd.DatetimeIndex(tr.index.get_level_values('datetime'));ev_dates=pd.DatetimeIndex(ev.index.get_level_values('datetime'))
train_mask=tr[('meta','eligible')].to_numpy(bool)&tr[('meta','cap_binding')].to_numpy(bool)&pd.to_datetime(tr[('meta','label_available_at')],utc=True).lt(pd.Timestamp('2021-01-01T00:00:00',tz='America/New_York').tz_convert('UTC')).to_numpy()&np.isfinite(tr[('label','local_fee_net_advantage')].to_numpy(float))
eval_mask=ev[('meta','eligible')].to_numpy(bool)&ev[('meta','cap_binding')].to_numpy(bool)&pd.to_datetime(ev[('meta','label_available_at')],utc=True).lt(pd.Timestamp('2023-01-01T00:00:00',tz='America/New_York').tz_convert('UTC')).to_numpy()&np.isfinite(ev[('label','local_fee_net_advantage')].to_numpy(float))
train=tr.iloc[np.flatnonzero(train_mask)];evaluation=ev.iloc[np.flatnonzero(eval_mask)];dates=pd.DatetimeIndex(evaluation.index.get_level_values('datetime'))
assert len(train)==plan['cohorts']['expected_train_rows']==239 and len(evaluation)==plan['cohorts']['expected_evaluation_rows']==301 and len(ev)==plan['cohorts']['evaluation_full_calendar_rows']==503
assert int((dates.year==2021).sum())==56 and int((dates.year==2022).sum())==245
base=plan['fixed_features']['baseline'];xtr=train['feature'][base].to_numpy(float);xev=evaluation['feature'][base].to_numpy(float);xtr_new=train['feature'][features].to_numpy(float);xev_new=evaluation['feature'][features].to_numpy(float);ytr=train[('label','local_fee_net_advantage')].to_numpy(float);yev=evaluation[('label','local_fee_net_advantage')].to_numpy(float);assert np.isfinite(xtr).all() and np.isfinite(xtr_new).all() and np.isfinite(yev).all()
preflight={'status':'same_state_cap_release_two_head_interaction_preflight_passed','plan_sha256':sha(card/'plan.json'),'full_calendar_rows':len(flat),'train_mature_binding':len(train),'evaluation_mature_binding':len(evaluation),'evaluation_2021':56,'evaluation_2022':245,'source_Recorder_object_loads':3,'model_fits':0,'native_accounts':0};(card/'preflight_result.json').write_text(json.dumps(preflight,indent=2)+'\n')
state={'status':'fit_started','plan_sha256':preflight['plan_sha256'],'fit_attempts':0,'fit_completed':0,'bootstrap_attempts':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
control=make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),Ridge(alpha=1.0));candidate=make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),Ridge(alpha=1.0))
for model,x in ((control,xtr),(candidate,xtr_new)):
 state['fit_attempts']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');model.fit(x,ytr);state['fit_completed']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
p0=control.predict(xev);p1=candidate.predict(xev_new);mean=float(ytr.mean());coef=float(candidate.named_steps['ridge'].coef_[-1]);scored=pd.DataFrame({'trade_date':dates,'label':yev,'additive_prediction':p0,'interaction_prediction':p1,'train_mean_prediction':mean});scored['gain_vs_additive']=(yev-p0)**2-(yev-p1)**2;scored['gain_vs_train_mean']=(yev-mean)**2-(yev-p1)**2
full=pd.DataFrame({'trade_date':ev_dates}).merge(scored,on='trade_date',how='left',validate='one_to_one');assert len(full)==503 and int(full.label.notna().sum())==301
metrics={'candidate_MSE':float(np.square(yev-p1).mean()),'additive_MSE':float(np.square(yev-p0).mean()),'train_mean_MSE':float(np.square(yev-mean).mean()),'added_standardized_coefficient':coef,'comparators':{}};gates={'coefficient_positive':bool(coef>0)};replicates={};indices=None
for key in ('additive','train_mean'):
 column='gain_vs_'+key;diff=scored[column];annual=[]
 for year in (2021,2022):
  take=scored.trade_date.dt.year.eq(year);y=scored.loc[take,'label'];pred=scored.loc[take,'interaction_prediction'];rho=float(pd.Series(pred.to_numpy(float)).corr(pd.Series(y.to_numpy(float)),method='spearman'));annual.append({'year':year,'rows':int(take.sum()),'mean_gain':float(diff.loc[take].mean()),'Spearman':rho})
 state['bootstrap_attempts']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');b=plan['information_gate']['bootstrap'];interval,idx,reps=full_calendar_bootstrap(full[column].to_numpy(float),mean_block=b['mean_block_sessions'],resamples=b['draws_each'],seed=b['seed'],alpha=b['one_sided_alpha_each']);assert interval['valid_replicates']==b['draws_each']
 if indices is None:indices=idx
 else:assert np.array_equal(indices,idx)
 replicates[key]=reps;metrics['comparators'][key]={'full_mean_gain':float(diff.mean()),'annual':annual,'bootstrap':interval};gates[key]={'full_positive':bool(diff.mean()>0),'both_years_positive':bool(all(z['mean_gain']>0 for z in annual)),'both_years_Spearman_positive':bool(all(z['Spearman']>0 for z in annual)),'corrected_lower_positive':bool(interval['lower_bound']>0)}
passed=bool(gates['coefficient_positive'] and all(all(v.values()) for v in gates.values() if isinstance(v,dict)))
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();dataset.config(dump_all=True,recursive=True);rec.save_objects(artifact_path='inputs',plan=plan,preflight=preflight,dataset=dataset);rec.save_objects(artifact_path='models',additive=control,interaction=candidate);rec.save_objects(artifact_path='predictions',scored=scored);rec.save_objects(artifact_path='analysis',full_calendar=full,metrics=metrics,gates=gates,bootstrap_replicates=replicates,bootstrap_indices=indices,information_gate_pass=passed);rec.log_metrics(fits=2,native_accounts=0,information_gate_pass=int(passed),gain_vs_additive=metrics['comparators']['additive']['full_mean_gain']);rid,eid=rec.id,str(rec.experiment_id)
state['status']='finished';(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
def ready(x):
 if isinstance(x,dict):return {str(k):ready(v) for k,v in x.items()}
 if isinstance(x,(list,tuple)):return [ready(v) for v in x]
 if isinstance(x,np.generic):return x.item()
 return x
summary={'experiment':plan['experiment'],'decision':'conditional_alpha_tail_cap_release_information_pass_needs_0bp_account' if passed else 'not_promoted_fixed_alpha_tail_cap_release_information_failed','objective_achieved':False,'plan_sha256':preflight['plan_sha256'],'preflight':preflight,'metrics':metrics,'gates':gates,'information_gate_pass':passed,'new_model_fits':2,'new_bootstraps':2,'new_native_accounts':0,'historical_independence':False,'recorder_id':rid,'experiment_id':eid,'recorder_uri':str(card/'qlib_recorder'),'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','models/*','predictions/*','analysis/*']},'limitations':['The target is a local one-session common-reset action value, not closed-loop strategy wealth.','2021/2022 history is reused development evidence; no independent 2023+ strategy evaluation.','A changed own-state account would generate different caps and risk states and requires separately frozen full-account proof.']};summary=ready(summary);(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid);settings=_settings(root);remote=_ssh(settings['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0];assert remote==evidence['sha256'];summary['NAS_evidence']=evidence;summary['NAS_readback']={'sha256':remote,'verified':True};(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
print(json.dumps({'decision':summary['decision'],'preflight':preflight,'metrics':metrics,'gates':gates,'recorder_id':rid,'NAS_sha256':remote},ensure_ascii=False,indent=2))


{
  "decision": "not_promoted_fixed_alpha_tail_cap_release_information_failed",
  "preflight": {
    "status": "same_state_cap_release_two_head_interaction_preflight_passed",
    "plan_sha256": "babaa29a86a742abf136f56bacef36209c102637d3fabec4f4a83c8ad108a1b5",
    "full_calendar_rows": 1061,
    "train_mature_binding": 239,
    "evaluation_mature_binding": 301,
    "evaluation_2021": 56,
    "evaluation_2022": 245,
    "source_Recorder_object_loads": 3,
    "model_fits": 0,
    "native_accounts": 0
  },
  "metrics": {
    "candidate_MSE": 6.369592282980019e-06,
    "additive_MSE": 6.368773792168581e-06,
    "train_mean_MSE": 6.330007046203221e-06,
    "added_standardized_coefficient": -0.0005256794553080371,
    "comparators": {
      "additive": {
        "full_mean_gain": -8.184908114375448e-10,
        "annual": [
          {
            "year": 2021,
            "rows": 56,
            "mean_gain": -3.513315191031759e-08,
            "Spearman": -0.05030758714969242
          },
 

## 中文结论

训练受限239、评价2021/2022为56/245；alpha×(1−tail)系数−0.00052568与预定正号相反，MSE6.369592e−6差于加性对照6.368774e−6和训练均值6.330007e−6，两项区块下界均负。固定信息门失败，2拟合2区块0账户；来源字段机械修复与NAS冷读留证，不按年份／符号救援。